# 05G — Observation-Process Proxy Sensitivity Audit

**Build ID:** `05G_V1_OBSERVATION_PROCESS_PROXY_SENSITIVITY`

05F successfully serialized the primary artifact. Its gain importance shows that several pre-anchor observation-process fields are highly ranked, including observation indices, previous-interval duration, history span, and support-weight sums.

These are not necessarily target leakage, but they may encode NIROPS observation cadence or measurement support rather than wildfire/fireline physics.

05G therefore compares the saved primary OOF model against two fixed-parameter sensitivity models using the same incident-held-out outer folds:

- `PHYSICAL_SENSITIVITY`: full feature set minus observation-process proxy fields
- `PROCESS_ONLY`: only the observation-process proxy fields

No tuning, calibration, threshold selection, or fold changes occur.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05G_V1_OBSERVATION_PROCESS_PROXY_SENSITIVITY"

from pathlib import Path
import sys
import subprocess
import importlib.util
import gc
import json
import time
import re

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
COLUMN_MANIFEST_PATH = PROJECT_ROOT / "outputs" / "modeling_table_assembly_v1" / "MODELING_COLUMN_MANIFEST_V1.csv"
PRIMARY_OOF_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "observation_process_proxy_sensitivity_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "observation_process_proxy_sensitivity_v1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, UNIT_DESIGN_PATH, FEATURE_SETS_PATH, COLUMN_MANIFEST_PATH, PRIMARY_OOF_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
OUTER_FOLDS = 5
MODEL_SEED = 20260825
PRED_EPS = 1e-6
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260825

FIXED_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 150,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
}

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "lightgbm": "lightgbm",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from lightgbm import LGBMClassifier

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)

## 1. Identify observation-process proxy features

In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    FEATURE_SETS = json.load(f)

full_features = FEATURE_SETS["E_FULL_EXTENDED"]
assert len(full_features) == 803

manifest = pd.read_csv(COLUMN_MANIFEST_PATH)

name_candidates = [c for c in ["column", "column_name", "feature", "feature_name"] if c in manifest.columns]
if not name_candidates:
    raise RuntimeError(f"No feature-name column found in manifest: {manifest.columns.tolist()}")
NAME_COL = name_candidates[0]

source_candidates = [c for c in ["source", "source_table", "feature_source"] if c in manifest.columns]
SOURCE_COL = source_candidates[0] if source_candidates else None

proxy_tokens = [
    "observation_index",
    "previous_interval_hours",
    "history_span_hours",
    "weight_sum",
    "observation_count",
    "n_observations",
    "history_count",
]

m = manifest[manifest[NAME_COL].isin(set(full_features))].copy()

if SOURCE_COL is not None:
    approach_mask = m[SOURCE_COL].astype(str).str.lower().eq("approach")
else:
    approach_mask = m[NAME_COL].astype(str).str.startswith("exact_")

proxy_mask = pd.Series(False, index=m.index)
for token in proxy_tokens:
    proxy_mask = proxy_mask | m[NAME_COL].astype(str).str.lower().str.contains(re.escape(token), regex=True)

proxy_features = sorted(
    m.loc[approach_mask & proxy_mask, NAME_COL].astype(str).tolist()
)

known_expected = [
    "exact_t0_observation_index",
    "exact_previous_interval_hours",
    "exact_history_span_hours",
]

for c in known_expected:
    if c in full_features:
        assert c in proxy_features, c

assert len(proxy_features) > 0

proxy_set = set(proxy_features)
physical_features = [c for c in full_features if c not in proxy_set]

assert len(physical_features) + len(proxy_features) == len(full_features)

proxy_audit = pd.DataFrame({"excluded_proxy_feature": proxy_features})
display(proxy_audit)

print("Full:", len(full_features))
print("Proxy:", len(proxy_features))
print("Physical sensitivity:", len(physical_features))

proxy_audit.to_csv(
    OUTPUT_ROOT / "OBSERVATION_PROCESS_PROXY_FEATURES_V1.csv",
    index=False,
)

## 2. Load frozen design and primary OOF

In [ ]:
design = pd.read_parquet(UNIT_DESIGN_PATH)
primary_saved = pd.read_parquet(PRIMARY_OOF_PATH)

assert len(design) == EXPECTED_UNITS
assert design["unit_uid"].is_unique
assert design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert len(primary_saved) == EXPECTED_UNITS
assert primary_saved["unit_uid"].is_unique

base = design[
    ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]
].copy()

primary = base.merge(
    primary_saved[
        ["unit_uid", "_irwin_norm", "hold_label", "outer_fold", "pred_prob"]
    ],
    on=["unit_uid", "_irwin_norm"],
    suffixes=("_design", "_saved"),
    validate="one_to_one",
)

assert (primary["hold_label_design"] == primary["hold_label_saved"]).all()
assert (primary["outer_fold_design"] == primary["outer_fold_saved"]).all()

primary = primary.rename(
    columns={
        "hold_label_design": "hold_label",
        "outer_fold_design": "outer_fold",
    }
).drop(columns=["hold_label_saved", "outer_fold_saved"])

print("Primary alignment: PASS")

## 3. Run fixed OOF sensitivity models

In [ ]:
def make_model(seed):
    params = dict(FIXED_PARAMS)
    params["random_state"] = int(seed)
    return LGBMClassifier(**params)

def run_fixed_oof(feature_list, experiment_id, seed_offset):
    feature_table = pd.read_parquet(
        MASTER_PATH,
        columns=["unit_uid", *feature_list],
    )

    frame = base.merge(
        feature_table,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    del feature_table
    gc.collect()

    X = frame[feature_list].to_numpy(dtype=np.float32, na_value=np.nan)
    y = frame["hold_label"].to_numpy(np.int8)
    folds = frame["outer_fold"].to_numpy(int)

    oof_prob = np.full(len(frame), np.nan, dtype=np.float64)
    runtime_rows = []

    for fold in range(OUTER_FOLDS):
        train_idx = np.flatnonzero(folds != fold)
        test_idx = np.flatnonzero(folds == fold)

        model = make_model(MODEL_SEED + seed_offset + fold)

        t0 = time.time()
        model.fit(X[train_idx], y[train_idx])
        elapsed = time.time() - t0

        oof_prob[test_idx] = model.predict_proba(X[test_idx])[:, 1]

        runtime_rows.append({
            "experiment_id": experiment_id,
            "outer_fold": fold,
            "train_units": len(train_idx),
            "test_units": len(test_idx),
            "features": len(feature_list),
            "fit_seconds": elapsed,
        })

        print(experiment_id, "fold", fold, "fit_min", round(elapsed/60.0, 3))

        del model
        gc.collect()

    assert np.isfinite(oof_prob).all()

    out = frame[
        ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]
    ].copy()
    out["pred_prob"] = oof_prob
    out["experiment_id"] = experiment_id

    return out, pd.DataFrame(runtime_rows)

physical_oof, physical_runtime = run_fixed_oof(
    physical_features,
    "PHYSICAL_SENSITIVITY",
    10_000,
)

process_oof, process_runtime = run_fixed_oof(
    proxy_features,
    "PROCESS_ONLY",
    20_000,
)

physical_oof.to_parquet(
    PROCESSED_ROOT / "PHYSICAL_SENSITIVITY_OOF_V1.parquet",
    index=False,
)

process_oof.to_parquet(
    PROCESSED_ROOT / "PROCESS_ONLY_OOF_V1.parquet",
    index=False,
)

runtime = pd.concat([physical_runtime, process_runtime], ignore_index=True)
runtime.to_csv(OUTPUT_ROOT / "SENSITIVITY_RUNTIME_V1.csv", index=False)

## 4. Metrics and paired incident bootstrap

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), PRED_EPS, 1.0 - PRED_EPS)

def logloss_vec(y, p):
    y = np.asarray(y, dtype=float)
    p = clip_prob(p)
    return -(y*np.log(p) + (1-y)*np.log(1-p))

def incident_metrics(frame):
    rows = []
    for incident, g in frame.groupby("_irwin_norm", sort=False):
        y = g["hold_label"].to_numpy(int)
        p = g["pred_prob"].to_numpy(float)
        mixed = np.unique(y).size == 2
        rows.append({
            "_irwin_norm": incident,
            "units": len(g),
            "brier_sum": float(np.sum((p-y)**2)),
            "logloss_sum": float(np.sum(logloss_vec(y,p))),
            "brier": float(np.mean((p-y)**2)),
            "log_loss": float(np.mean(logloss_vec(y,p))),
            "mixed_class": mixed,
            "pr_auc": float(average_precision_score(y,p)) if mixed else np.nan,
            "roc_auc": float(roc_auc_score(y,p)) if mixed else np.nan,
        })
    return pd.DataFrame(rows)

def summarize(frame):
    y = frame["hold_label"].to_numpy(int)
    p = frame["pred_prob"].to_numpy(float)
    inc = incident_metrics(frame)
    mixed = inc["mixed_class"]

    return {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "mean_pred": float(np.mean(p)),
        "pr_auc": float(average_precision_score(y,p)),
        "roc_auc": float(roc_auc_score(y,p)),
        "brier": float(brier_score_loss(y,p)),
        "log_loss": float(log_loss(y, clip_prob(p), labels=[0,1])),
        "incident_macro_brier": float(inc["brier"].mean()),
        "incident_macro_log_loss": float(inc["log_loss"].mean()),
        "mixed_macro_pr_auc": float(inc.loc[mixed,"pr_auc"].mean()),
        "mixed_macro_roc_auc": float(inc.loc[mixed,"roc_auc"].mean()),
    }, inc

primary_eval = primary[
    ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold", "pred_prob"]
].copy()

primary_metrics, primary_inc = summarize(primary_eval)
physical_metrics, physical_inc = summarize(physical_oof)
process_metrics, process_inc = summarize(process_oof)

metrics = pd.DataFrame([
    {"experiment_id": "PRIMARY_FULL", "features": len(full_features), **primary_metrics},
    {"experiment_id": "PHYSICAL_SENSITIVITY", "features": len(physical_features), **physical_metrics},
    {"experiment_id": "PROCESS_ONLY", "features": len(proxy_features), **process_metrics},
])

display(metrics)
metrics.to_csv(
    OUTPUT_ROOT / "OBSERVATION_PROCESS_SENSITIVITY_METRICS_V1.csv",
    index=False,
)

incident_ids = np.array(sorted(primary_eval["_irwin_norm"].unique()))

aligned = {
    "PRIMARY_FULL": primary_inc.set_index("_irwin_norm").loc[incident_ids],
    "PHYSICAL_SENSITIVITY": physical_inc.set_index("_irwin_norm").loc[incident_ids],
    "PROCESS_ONLY": process_inc.set_index("_irwin_norm").loc[incident_ids],
}

rng = np.random.default_rng(BOOTSTRAP_SEED)
rows = []

loss_metrics = {"pooled_brier", "pooled_log_loss", "incident_macro_brier", "incident_macro_log_loss"}

for rep in range(BOOTSTRAP_REPS):
    draw = rng.integers(0, EXPECTED_INCIDENTS, size=EXPECTED_INCIDENTS)
    mult = np.bincount(draw, minlength=EXPECTED_INCIDENTS).astype(float)

    repm = {}

    for exp, x in aligned.items():
        units = x["units"].to_numpy(float)
        total_units = np.sum(mult * units)
        mixed = x["mixed_class"].to_numpy(bool)
        mm = mult[mixed]

        repm[exp] = {
            "pooled_brier": float(np.sum(mult*x["brier_sum"].to_numpy(float))/total_units),
            "pooled_log_loss": float(np.sum(mult*x["logloss_sum"].to_numpy(float))/total_units),
            "incident_macro_brier": float(np.sum(mult*x["brier"].to_numpy(float))/np.sum(mult)),
            "incident_macro_log_loss": float(np.sum(mult*x["log_loss"].to_numpy(float))/np.sum(mult)),
            "mixed_macro_pr_auc": float(np.sum(mm*x.loc[mixed,"pr_auc"].to_numpy(float))/np.sum(mm)),
            "mixed_macro_roc_auc": float(np.sum(mm*x.loc[mixed,"roc_auc"].to_numpy(float))/np.sum(mm)),
        }

    for alt in ["PHYSICAL_SENSITIVITY", "PROCESS_ONLY"]:
        for metric in repm["PRIMARY_FULL"]:
            rows.append({
                "rep": rep,
                "alternative": alt,
                "metric": metric,
                "delta_alt_minus_primary": repm[alt][metric] - repm["PRIMARY_FULL"][metric],
            })

boot = pd.DataFrame(rows)

summary_rows = []
for (alt, metric), g in boot.groupby(["alternative", "metric"]):
    d = g["delta_alt_minus_primary"].to_numpy(float)
    better = d < 0 if metric in loss_metrics else d > 0
    summary_rows.append({
        "alternative": alt,
        "metric": metric,
        "mean_delta": float(d.mean()),
        "ci_low": float(np.quantile(d,0.025)),
        "ci_high": float(np.quantile(d,0.975)),
        "probability_alternative_better": float(np.mean(better)),
    })

bootstrap_summary = pd.DataFrame(summary_rows)
display(bootstrap_summary)

bootstrap_summary.to_csv(
    OUTPUT_ROOT / "OBSERVATION_PROCESS_SENSITIVITY_BOOTSTRAP_V1.csv",
    index=False,
)

## 5. Final checks and report

In [ ]:
checks = pd.DataFrame([
    {"check": "exact_803_primary_features", "pass": len(full_features) == 803},
    {"check": "proxy_feature_set_nonempty", "pass": len(proxy_features) > 0},
    {
        "check": "known_top_observation_process_features_captured",
        "pass": all((c not in full_features) or (c in proxy_features) for c in known_expected),
    },
    {
        "check": "physical_plus_proxy_reconstructs_full_set",
        "pass": len(physical_features) + len(proxy_features) == len(full_features),
    },
    {"check": "exact_10_new_outer_fits", "pass": len(runtime) == 10},
    {
        "check": "all_new_oof_tables_exact_304143_units",
        "pass": len(physical_oof) == EXPECTED_UNITS and len(process_oof) == EXPECTED_UNITS,
    },
    {"check": "same_frozen_outer_incident_folds", "pass": True},
    {"check": "exact_fixed_05c_hyperparameters_no_tuning", "pass": True},
    {
        "check": "paired_incident_bootstrap_5000_complete",
        "pass": boot["rep"].nunique() == BOOTSTRAP_REPS,
    },
    {"check": "no_threshold_selection", "pass": True},
])

display(checks)

PASS = bool(checks["pass"].all())

print()
print(
    "05G OBSERVATION-PROCESS PROXY SENSITIVITY:",
    "PASS / READY FOR SCIENTIFIC ROBUSTNESS INTERPRETATION"
    if PASS else "REVIEW REQUIRED",
)

checks.to_csv(
    OUTPUT_ROOT / "FINAL_OBSERVATION_PROCESS_PROXY_CHECKS_V1.csv",
    index=False,
)

def md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report_lines = [
    "# 05G Observation-Process Proxy Sensitivity Audit",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    "## Excluded observation-process fields",
    "",
    md_table(proxy_audit),
    "",
    "## Metrics",
    "",
    md_table(metrics),
    "",
    "## Paired incident-bootstrap differences versus primary",
    "",
    md_table(bootstrap_summary),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / READY FOR SCIENTIFIC ROBUSTNESS INTERPRETATION`" if PASS else "`REVIEW REQUIRED`",
]

REPORT_PATH = OUTPUT_ROOT / "OBSERVATION_PROCESS_PROXY_SENSITIVITY_REPORT_V1.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved:", REPORT_PATH)